In [8]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score


# =========================
# 2. LOAD DATA
# =========================
df = pd.read_excel(
    r'C:\Users\FA23-BCS-041.CUI\Desktop\datasets\default of credit card clients.xls',
    engine='xlrd'
)


# =========================
# 3. CLEAN DATA (IMPORTANT FIX HERE)
# =========================

# Remove useless column safely
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

# Convert EVERYTHING to numeric (fix scaling error root cause)
df = df.apply(pd.to_numeric, errors='coerce')

# Drop rows with missing values
df = df.dropna()

# Ensure target is integer
df['Y'] = df['Y'].astype(int)


# =========================
# 4. SPLIT FEATURES & TARGET
# =========================
X = df.drop('Y', axis=1)
y = df['Y']


# =========================
# 5. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# =========================
# 6. FEATURE SCALING (NOW SAFE)
# =========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# =========================
# 7. MODEL TRAINING
# =========================
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]


# =========================
# 8. BUSINESS COST FUNCTION
# =========================
COST_FN = 100000   # missed default (worst case)
COST_FP = 10000    # rejected good customer


# =========================
# 9. THRESHOLD OPTIMIZATION
# =========================
thresholds = np.arange(0.1, 0.9, 0.01)

best_threshold = 0
best_cost = float('inf')

for t in thresholds:
    preds = (probs >= t).astype(int)

    FN = np.sum((preds == 0) & (y_test == 1))
    FP = np.sum((preds == 1) & (y_test == 0))

    cost = (FN * COST_FN) + (FP * COST_FP)

    if cost < best_cost:
        best_cost = cost
        best_threshold = t


# =========================
# 10. FINAL RESULTS
# =========================
final_preds = (probs >= best_threshold).astype(int)

print("\nBEST THRESHOLD:", best_threshold)
print("MINIMUM COST:", best_cost)

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, final_preds))

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, final_preds))

print("\nROC AUC SCORE:")
print(roc_auc_score(y_test, probs))


BEST THRESHOLD: 0.1
MINIMUM COST: 49570000

CONFUSION MATRIX:
[[ 980 3707]
 [ 125 1188]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       0.89      0.21      0.34      4687
           1       0.24      0.90      0.38      1313

    accuracy                           0.36      6000
   macro avg       0.56      0.56      0.36      6000
weighted avg       0.75      0.36      0.35      6000


ROC AUC SCORE:
0.7269867506354779
